In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=None,
    stop_sequences=None,
    tools=None,
    thinking=False,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if thinking:
        # Adaptive thinking is the current API: Claude decides when and how much
        # to think (the legacy {"type": "enabled", "budget_tokens": N} mode is
        # rejected with a 400 on recent models). display="summarized" opts into
        # readable thinking text — the default on claude-sonnet-5 is "omitted",
        # which returns thinking blocks with an empty thinking field.
        params["thinking"] = {"type": "adaptive", "display": "summarized"}

    if system:
        params["system"] = system

    if temperature is not None:
        params["temperature"] = temperature

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if tools:
        params["tools"] = tools

    return client.messages.create(**params)


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

In [3]:
# Ask a multistep reasoning question with thinking enabled — the response
# starts with a ThinkingBlock (summarized reasoning + signature) followed by
# the final TextBlock. Adaptive thinking is steerable per message: phrases like
# "think hard" encourage it, "answer directly" suppresses it
messages = []

add_user_message(
    messages,
    "What is the greatest common divisor of 1071 and 462? "
    "Please think hard before responding.",
)

response = chat(messages, thinking=True)
response

Message(id='msg_011Cd6jxwxqbKDW49kMt4AkC', container=None, content=[ThinkingBlock(signature='EqwDCokBCA8YAipABqzVFNmGgmvG29Hp9f8EorqIlIKp1Qbvs0NlsJdPG7QCqGbmqRj5Ba0Wo1Z8VYNjBE3Go5+PviXYpdVFYgzFhTIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiRmZDAzY2UwZC02YzkyLTQ2NWEtOWNmZi1iMDQ4MTZhODlhNGISDBYUSzaSjknucMQLwRoMQuHqkXpGxEYthk8dIjAyCAw3Si54MUM+WfCzbSKcpPX8WaknUG99irvOQe23NTRybN8gvnHvgWJlsi354pcqzwE+Fdx1mhoC97VJAbhq7pdDzxrFQ7xzoq8VHFcCKmidUu76oCyWsmzrdylQiqsb2TdXy80+7NlZc3ttpxUEptR3Ugr1eWCWFBKNxS1hYPUdEkw59mY2aAHJvId5+iLsAzSD93fbtRl1CTPst7J20nbH9B7OFqroyRdweGFYmjYkGhxw8KMJMOkG0N4SXcA7G1GNd9FrzgWAB6i4j4IUM9RcsuBMYx7FNa34qZlMscccFUuRzqoAZYhn+8XSH5Kxtb+aOPJD2OEUsj0QsRWi4oYYAQ==', thinking="I'm working through the Euclidean algorithm to find the GCD of 1071 and 462, which gives me 21 after three steps. Let me verify this is correct by checking that 1071 and 462 both divide evenly by 21, leaving 51 and 22 respectively, and confirming those two numbers share no common factors.", type='thinking'), Text

In [4]:
# You are billed for the full internal reasoning, not the summary you see —
# usage.output_tokens_details.thinking_tokens breaks that spend down
details = response.usage.output_tokens_details

print(f"Output tokens (billed): {response.usage.output_tokens}")
print(f"...of which thinking:   {details.thinking_tokens if details else 0}")

Output tokens (billed): 418
...of which thinking:   118


In [5]:
# Multi-turn rule: pass thinking blocks back exactly as received —
# add_assistant_message appends response.content unchanged, so the signed
# thinking block stays valid for the follow-up request
add_assistant_message(messages, response)
add_user_message(messages, "Now find the least common multiple of the same two numbers.")

followup = chat(messages, thinking=True)
print(text_from_message(followup))

# Finding LCM(1071, 462)

## Using the GCD-LCM Relationship

The relationship between GCD and LCM states:
$$\text{LCM}(a,b) = \frac{a \times b}{\text{GCD}(a,b)}$$

From the previous calculation, we know:
- $\gcd(1071, 462) = 21$
- $1071 = 21 \times 51$
- $462 = 21 \times 22$

## Calculation

Since $51$ and $22$ share no common factors (verified earlier), I can compute the LCM efficiently:

$$\text{LCM}(1071, 462) = 21 \times 51 \times 22$$

**Step 1:** Multiply 51 × 22
$$51 \times 22 = 1122$$

**Step 2:** Multiply by 21
$$1122 \times 21 = 23562$$

## Verification

Check that 23562 is divisible by both original numbers:
- $23562 \div 1071 = 22$ ✓
- $23562 \div 462 = 51$ ✓

## Answer
$$\text{LCM}(1071, 462) = \boxed{23562}$$
